In [ ]:
import gc
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

train_path = Path("processed/train_final_scaled.csv")
test_path = Path("processed/test_final_scaled.csv")
train_sel_path = Path("processed/train_final_selected.csv")
scaler_path = Path("processed/minmax_scaler_final.joblib")

assert train_path.exists() and test_path.exists(), "Processed datasets not found!"
Path("models").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

# 11 Multiclass labels: raw 1..11 -> remapped 0..10
CLASS_NAMES = [
    "Benign",              # 0 (raw 1)
    "FTP-BruteForce",      # 1 (raw 2)
    "SSH-BruteForce",      # 2 (raw 3)
    "DDoS-HOIC",           # 3 (raw 4)
    "Bot",                 # 4 (raw 5)
    "DoS-GoldenEye",       # 5 (raw 6)
    "DoS-Slowloris",       # 6 (raw 7)
    "DDoS-LOIC-UDP",       # 7 (raw 8)
    "Brute Force -Web",    # 8 (raw 9)
    "Brute Force -XSS",    # 9 (raw 10)
    "SQL Injection",       # 10 (raw 11)
]
NUM_CLASSES = len(CLASS_NAMES)

# Φόρτωση scaled datasets (float32 για εξοικονόμηση RAM)
columns = pd.read_csv(train_path, nrows=0).columns.tolist()
features = [c for c in columns if c != "Label"]
dtypes = {c: "float32" for c in features}
dtypes["Label"] = "int16"

train_df = pd.read_csv(train_path, dtype=dtypes)
test_df = pd.read_csv(test_path, dtype=dtypes)

X_train = np.ascontiguousarray(train_df[features].to_numpy(dtype=np.float32))
X_test = np.ascontiguousarray(test_df[features].to_numpy(dtype=np.float32))
y_train = (train_df["Label"].to_numpy(dtype=np.int64) - 1)
y_test = (test_df["Label"].to_numpy(dtype=np.int64) - 1)

del train_df, test_df
gc.collect()

assert set(np.unique(y_train)) == set(range(NUM_CLASSES))
assert set(np.unique(y_test)) == set(range(NUM_CLASSES))

# Επαλήθευση φυσικών υπογραφών (Dst Port / Protocol / Flow Duration) από τα unscaled δεδομένα
profile_cols = ["Dst Port", "Protocol", "Flow Duration", "Label"]
sel_df = pd.read_csv(train_sel_path, usecols=profile_cols)

dist_rows = []
for cls_id, cname in enumerate(CLASS_NAMES):
    raw_lbl = cls_id + 1
    tr_cnt = int((y_train == cls_id).sum())
    te_cnt = int((y_test == cls_id).sum())
    sub = sel_df[sel_df["Label"] == raw_lbl]
    top_port = int(sub["Dst Port"].mode().iloc[0])
    top_proto = int(sub["Protocol"].mode().iloc[0])
    proto_str = {6: "TCP(6)", 17: "UDP(17)", 0: "HOPOPT(0)"}.get(top_proto, str(top_proto))
    dist_rows.append({
        "Class_ID": cls_id,
        "Raw_Label": raw_lbl,
        "Attack_Type": cname,
        "Train_Count": tr_cnt,
        "Train_Pct": round(100.0 * tr_cnt / len(y_train), 4),
        "Test_Count": te_cnt,
        "Test_Pct": round(100.0 * te_cnt / len(y_test), 4),
        "Dominant_Dst_Port": top_port,
        "Dominant_Protocol": proto_str,
        "Median_Duration_us": round(float(sub["Flow Duration"].median()), 1),
    })

del sel_df
gc.collect()

dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv("results/q6_multiclass_class_distribution.csv", index=False)

print("=== MULTICLASS DATASET (11 CLASSES) ===")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape} | Imbalance ratio (Benign:SQLi): {dist_df.iloc[0]['Train_Count'] / dist_df.iloc[-1]['Train_Count']:,.0f}:1\n")
print(dist_df.to_string(index=False))

# Διάγραμμα κατανομής κλάσεων (log-scale)
fig, ax = plt.subplots(figsize=(12, 5.2))
x_pos = np.arange(NUM_CLASSES)
w = 0.38
bars_tr = ax.bar(x_pos - w/2, dist_df["Train_Count"], width=w, label="Training Set (715,505)", color="#1f77b4")
bars_te = ax.bar(x_pos + w/2, dist_df["Test_Count"], width=w, label="Testing Set (178,877)", color="#ff7f0e")
ax.set_yscale("log")
ax.set_xticks(x_pos)
ax.set_xticklabels(CLASS_NAMES, rotation=28, ha="right", fontsize=9.5)
ax.set_ylabel("Πλήθος Ροών (λογαριθμική κλίμακα)")
ax.set_title("Κατανομή των 11 Κλάσεων σε Training & Testing Set (Ακραία Ανισορροπία 89.855:1)", fontweight="bold")
for b in bars_tr:
    h = int(b.get_height())
    ax.text(b.get_x() + b.get_width()/2, h * 1.18, f"{h:,}", ha="center", va="bottom", fontsize=7.5, rotation=0)
ax.set_ylim(1, dist_df["Train_Count"].max() * 3.5)
ax.legend(loc="upper right")
ax.grid(axis="y", which="both", linestyle="--", alpha=0.45)
plt.tight_layout()
plt.savefig("results/q6_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
def evaluate_multiclass(name, y_true, y_pred):
    acc = float(accuracy_score(y_true, y_pred))
    macro_p = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    macro_r = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    weighted_p = float(precision_score(y_true, y_pred, average="weighted", zero_division=0))
    weighted_r = float(recall_score(y_true, y_pred, average="weighted", zero_division=0))
    weighted_f1 = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))

    # Παραγόμενες δυαδικές μετρικές (Benign=0 vs Any Attack>0)
    y_true_bin = (y_true != 0).astype(int)
    y_pred_bin = (y_pred != 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    bin_fpr = float(fp / (fp + tn))
    bin_recall = float(tp / (tp + fn))

    return {
        "Model": name,
        "Accuracy": acc,
        "Macro_Precision": macro_p,
        "Macro_Recall": macro_r,
        "Macro_F1": macro_f1,
        "Weighted_Precision": weighted_p,
        "Weighted_Recall": weighted_r,
        "Weighted_F1": weighted_f1,
        "Binary_Attack_Recall": bin_recall,
        "Binary_FPR": bin_fpr,
        "Binary_FP": int(fp),
        "Binary_FN": int(fn),
    }

# Εκπαίδευση Multiclass Random Forest
t0 = time.perf_counter()
rf_mc = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
)
rf_mc.fit(X_train, y_train)
rf_train_time = time.perf_counter() - t0

# Αποθήκευση μοντέλου & μέτρηση μεγέθους
rf_model_path = Path("models/rf_multiclass.joblib")
joblib.dump(rf_mc, rf_model_path)
rf_size_kb = rf_model_path.stat().st_size / 1024
rf_nodes = int(sum(est.tree_.node_count for est in rf_mc.estimators_))
rf_mean_depth = float(np.mean([est.get_depth() for est in rf_mc.estimators_]))

# Πρόβλεψη & Αξιολόγηση στο Testing Set
rf_preds = rf_mc.predict(X_test)
rf_metrics = evaluate_multiclass("Random Forest (Multiclass)", y_test, rf_preds)
rf_cm = confusion_matrix(y_test, rf_preds, labels=np.arange(NUM_CLASSES))

print("=== MULTICLASS RANDOM FOREST RESULTS ===")
print(f"Training time:       {rf_train_time:.2f} s")
print(f"Model size:          {rf_size_kb:.2f} KB ({rf_size_kb/1024:.3f} MB) | Nodes: {rf_nodes:,} | Mean depth: {rf_mean_depth:.1f}")
print(f"Accuracy:            {rf_metrics['Accuracy']:.6f}")
print(f"Macro F1:            {rf_metrics['Macro_F1']:.6f}  (P={rf_metrics['Macro_Precision']:.4f}, R={rf_metrics['Macro_Recall']:.4f})")
print(f"Weighted F1:         {rf_metrics['Weighted_F1']:.6f}")
print(f"Derived Binary FPR:  {rf_metrics['Binary_FPR']:.6f}  (FP={rf_metrics['Binary_FP']}, FN={rf_metrics['Binary_FN']})")
print("\nPer-class Classification Report (RF):")
print(classification_report(y_test, rf_preds, target_names=CLASS_NAMES, digits=4, zero_division=0))


In [ ]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 85% / 15% Stratified Train / Validation split (διατηρεί όλες τις 11 κλάσεις, συμπεριλαμβανομένου SQLi: 6 train / 1 val)
train_idx, val_idx = train_test_split(
    np.arange(len(X_train)),
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)

X_tr_mlp = X_train[train_idx]
y_tr_mlp = y_train[train_idx]
X_val_mlp = X_train[val_idx]
y_val_mlp = y_train[val_idx]

# Power-dampened inverse-frequency class weights (alpha=0.25)
class_counts = np.bincount(y_tr_mlp, minlength=NUM_CLASSES).astype(np.float64)
ALPHA = 0.25
class_weights = (len(y_tr_mlp) / (NUM_CLASSES * class_counts)) ** ALPHA
class_weights = (class_weights / class_weights.mean()).astype(np.float32)

class MulticlassIntrusionMLP(nn.Module):
    def __init__(self, input_features, num_classes=11):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.network(x)

mlp_mc = MulticlassIntrusionMLP(X_train.shape[1], NUM_CLASSES).to(device)
mlp_params = sum(p.numel() for p in mlp_mc.parameters() if p.requires_grad)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=device)
)
optimizer = torch.optim.Adam(mlp_mc.parameters(), lr=0.001)

EPOCHS = 20
BATCH_SIZE = 512
PATIENCE = 6
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("=== MULTICLASS MLP ARCHITECTURE ===")
print(mlp_mc)
print(f"Device: {device} | Trainable parameters: {mlp_params:,}")
print("Power-dampened class weights (alpha=0.25):", np.round(class_weights, 3))
print()

X_tr_t = torch.from_numpy(X_tr_mlp).to(device)
y_tr_t = torch.from_numpy(y_tr_mlp).to(device)
X_val_t = torch.from_numpy(X_val_mlp).to(device)

mlp_model_path = Path("models/mlp_multiclass_best.pt")
best_val_mf1 = -1.0
patience_cnt = 0
history = []

t_train_start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    t_ep = time.perf_counter()
    mlp_mc.train()
    perm = torch.randperm(len(X_tr_t), device=device)
    total_loss = 0.0

    for i in range(0, len(X_tr_t), BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        optimizer.zero_grad()
        logits = mlp_mc(X_tr_t[idx])
        loss = criterion(logits, y_tr_t[idx])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)

    scheduler.step()
    train_loss = total_loss / len(X_tr_t)

    mlp_mc.eval()
    with torch.inference_mode():
        val_preds = mlp_mc(X_val_t).argmax(dim=1).cpu().numpy()

    val_wf1 = float(f1_score(y_val_mlp, val_preds, average="weighted", zero_division=0))
    val_mf1 = float(f1_score(y_val_mlp, val_preds, average="macro", zero_division=0))
    ep_time = time.perf_counter() - t_ep

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_weighted_f1": val_wf1,
        "val_macro_f1": val_mf1,
        "epoch_time_s": ep_time,
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | Loss: {train_loss:.4f} | "
        f"Val W-F1: {val_wf1:.5f} | Val Macro-F1: {val_mf1:.5f} | Time: {ep_time:.2f}s",
        flush=True,
    )

    if val_mf1 > best_val_mf1:
        best_val_mf1 = val_mf1
        patience_cnt = 0
        torch.save(mlp_mc.state_dict(), mlp_model_path)
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}!")
            break

mlp_train_time = time.perf_counter() - t_train_start

# Φόρτωση καλύτερου μοντέλου & αξιολόγηση στο Testing Set
mlp_mc.load_state_dict(torch.load(mlp_model_path, map_location=device, weights_only=True))
mlp_mc.eval()
mlp_size_kb = mlp_model_path.stat().st_size / 1024

mlp_test_preds = []
with torch.inference_mode():
    for i in range(0, len(X_test), BATCH_SIZE):
        xb = torch.from_numpy(X_test[i:i+BATCH_SIZE]).to(device)
        mlp_test_preds.append(mlp_mc(xb).argmax(dim=1).cpu().numpy())
mlp_preds = np.concatenate(mlp_test_preds)

mlp_metrics = evaluate_multiclass("MLP (Multiclass)", y_test, mlp_preds)
mlp_cm = confusion_matrix(y_test, mlp_preds, labels=np.arange(NUM_CLASSES))

del X_tr_t, y_tr_t, X_val_t
gc.collect()

print("\n=== MULTICLASS MLP TEST RESULTS ===")
print(f"Training time:       {mlp_train_time:.2f} s ({len(history)} epochs, best Val Macro-F1={best_val_mf1:.5f})")
print(f"Model size:          {mlp_size_kb:.2f} KB ({mlp_params:,} params)")
print(f"Accuracy:            {mlp_metrics['Accuracy']:.6f}")
print(f"Macro F1:            {mlp_metrics['Macro_F1']:.6f}  (P={mlp_metrics['Macro_Precision']:.4f}, R={mlp_metrics['Macro_Recall']:.4f})")
print(f"Weighted F1:         {mlp_metrics['Weighted_F1']:.6f}")
print(f"Derived Binary FPR:  {mlp_metrics['Binary_FPR']:.6f}  (FP={mlp_metrics['Binary_FP']}, FN={mlp_metrics['Binary_FN']})")
print("\nPer-class Classification Report (MLP):")
print(classification_report(y_test, mlp_preds, target_names=CLASS_NAMES, digits=4, zero_division=0))


In [ ]:
# Πίνακας ανά κλάση (RF vs MLP)
rf_rep = classification_report(y_test, rf_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
mlp_rep = classification_report(y_test, mlp_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)

per_class_rows = []
for cname in CLASS_NAMES:
    per_class_rows.append({
        "Attack_Type": cname,
        "Test_Support": int(rf_rep[cname]["support"]),
        "RF_Precision": rf_rep[cname]["precision"],
        "RF_Recall": rf_rep[cname]["recall"],
        "RF_F1": rf_rep[cname]["f1-score"],
        "MLP_Precision": mlp_rep[cname]["precision"],
        "MLP_Recall": mlp_rep[cname]["recall"],
        "MLP_F1": mlp_rep[cname]["f1-score"],
        "Delta_F1_RF_minus_MLP": rf_rep[cname]["f1-score"] - mlp_rep[cname]["f1-score"],
    })
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv("results/q6_multiclass_per_class_metrics.csv", index=False)

print("=== PER-CLASS COMPARISON (RF vs MLP) ===")
print(per_class_df.round(4).to_string(index=False))

# --- Plot 2: Side-by-Side Row-Normalized 11x11 Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8.2))
short_names = [
    "Benign", "FTP-BF", "SSH-BF", "DDoS-HOIC", "Bot",
    "GoldenEye", "Slowloris", "LOIC-UDP", "Web-BF", "XSS", "SQLi"
]

for ax, cm, title in [
    (axes[0], rf_cm, f"Random Forest — Normalized CM (Macro-F1={rf_metrics['Macro_F1']:.4f})"),
    (axes[1], mlp_cm, f"Multiclass MLP — Normalized CM (Macro-F1={mlp_metrics['Macro_F1']:.4f})"),
]:
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0) * 100.0
    annot = np.empty_like(cm, dtype=object)
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            if cm[r, c] > 0:
                annot[r, c] = f"{cm_norm[r, c]:.1f}%\n({cm[r, c]:,})"
            else:
                annot[r, c] = ""
    sns.heatmap(
        cm_norm,
        annot=annot,
        fmt="",
        cmap="Blues",
        vmin=0,
        vmax=100,
        xticklabels=short_names,
        yticklabels=short_names,
        ax=ax,
        cbar_kws={"label": "Recall ανά πραγματική κλάση (%)"},
        annot_kws={"size": 7.2},
        linewidths=0.4,
        linecolor="#e0e0e0",
    )
    ax.set_xlabel("Predicted Class", fontsize=11)
    ax.set_ylabel("Actual Class", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", rotation=35)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("results/q6_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 3: Per-Class F1-Score Comparison (RF vs MLP) ---
fig, ax = plt.subplots(figsize=(12.5, 6.2))
y_pos = np.arange(NUM_CLASSES)
h = 0.37

b1 = ax.barh(y_pos + h/2, per_class_df["RF_F1"], height=h, label=f"Random Forest (Macro-F1={rf_metrics['Macro_F1']:.4f})", color="#1f77b4")
b2 = ax.barh(y_pos - h/2, per_class_df["MLP_F1"], height=h, label=f"Multiclass MLP (Macro-F1={mlp_metrics['Macro_F1']:.4f})", color="#ff7f0e")

for bar in b1:
    w_val = bar.get_width()
    ax.text(w_val + 0.008, bar.get_y() + bar.get_height()/2, f"{w_val:.3f}", va="center", ha="left", fontsize=8, color="#1f77b4", fontweight="bold")
for bar in b2:
    w_val = bar.get_width()
    ax.text(w_val + 0.008, bar.get_y() + bar.get_height()/2, f"{w_val:.3f}", va="center", ha="left", fontsize=8, color="#d95f02", fontweight="bold")

labels_with_sup = [f"{r['Attack_Type']} (n={r['Test_Support']:,})" for _, r in per_class_df.iterrows()]
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_with_sup, fontsize=9.5)
ax.invert_yaxis()
ax.set_xlim(0, 1.12)
ax.set_xlabel("F1-score", fontsize=11)
ax.set_title("Σύγκριση F1-score ανά Είδος Επίθεσης: Multiclass Random Forest vs MLP", fontsize=12.5, fontweight="bold")
ax.axvline(rf_metrics["Macro_F1"], color="#1f77b4", linestyle="--", alpha=0.55, linewidth=1.2)
ax.axvline(mlp_metrics["Macro_F1"], color="#ff7f0e", linestyle="--", alpha=0.55, linewidth=1.2)
ax.legend(loc="lower right", fontsize=9.5, frameon=True)
ax.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("results/q6_per_class_f1_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
N_SINGLE_INF = 2000
N_BATCH_REPS = 5
BATCH_BENCH = 512

# 1. Random Forest Inference Benchmark (n_jobs=1, ίδιο με Q3 & Q5)
rf_mc.set_params(n_jobs=1)
for _ in range(100):
    rf_mc.predict(X_test[:1])
for _ in range(20):
    rf_mc.predict(X_test[:BATCH_BENCH])

rf_single_ms = []
for i in range(N_SINGLE_INF):
    ts = time.perf_counter_ns()
    rf_mc.predict(X_test[i:i+1])
    rf_single_ms.append((time.perf_counter_ns() - ts) / 1_000_000)

rf_batch_times = []
for _ in range(N_BATCH_REPS):
    ts = time.perf_counter()
    for i in range(0, len(X_test), BATCH_BENCH):
        rf_mc.predict(X_test[i:i+BATCH_BENCH])
    rf_batch_times.append(time.perf_counter() - ts)

rf_inf_mean_ms = float(np.mean(rf_single_ms))
rf_inf_med_ms = float(np.median(rf_single_ms))
rf_inf_p95_ms = float(np.percentile(rf_single_ms, 95))
rf_batch_total_s = float(np.median(rf_batch_times))
rf_batch_thr = float(len(X_test) / rf_batch_total_s)
rf_amort_ms = float(rf_batch_total_s / len(X_test) * 1000.0)

# 2. Multiclass MLP Inference Benchmark (torch.inference_mode(), ίδιο με Q4)
mlp_mc.eval()
X_test_t = torch.from_numpy(X_test).to(device)
with torch.inference_mode():
    for _ in range(100):
        mlp_mc(X_test_t[:1])
    for _ in range(20):
        mlp_mc(X_test_t[:BATCH_BENCH])

    mlp_single_ms = []
    for i in range(N_SINGLE_INF):
        ts = time.perf_counter_ns()
        mlp_mc(X_test_t[i:i+1])
        mlp_single_ms.append((time.perf_counter_ns() - ts) / 1_000_000)

    mlp_batch_times = []
    for _ in range(N_BATCH_REPS):
        ts = time.perf_counter()
        for i in range(0, len(X_test_t), BATCH_BENCH):
            mlp_mc(X_test_t[i:i+BATCH_BENCH])
        mlp_batch_times.append(time.perf_counter() - ts)

mlp_inf_mean_ms = float(np.mean(mlp_single_ms))
mlp_inf_med_ms = float(np.median(mlp_single_ms))
mlp_inf_p95_ms = float(np.percentile(mlp_single_ms, 95))
mlp_batch_total_s = float(np.median(mlp_batch_times))
mlp_batch_thr = float(len(X_test_t) / mlp_batch_total_s)
mlp_amort_ms = float(mlp_batch_total_s / len(X_test_t) * 1000.0)

print("=== MULTICLASS INFERENCE BENCHMARK ===")
print(f"RF  (n_jobs=1):    Single Med={rf_inf_med_ms:.4f} ms | P95={rf_inf_p95_ms:.4f} ms | Batch Thr={rf_batch_thr:,.1f} flows/s ({rf_amort_ms:.5f} ms/flow)")
print(f"MLP (CPU {torch.get_num_threads()} thr.):  Single Med={mlp_inf_med_ms:.4f} ms | P95={mlp_inf_p95_ms:.4f} ms | Batch Thr={mlp_batch_thr:,.1f} flows/s ({mlp_amort_ms:.5f} ms/flow)")
print(f"Speedup MLP vs RF: Single-flow {rf_inf_med_ms/mlp_inf_med_ms:.1f}x faster | Batch throughput {mlp_batch_thr/rf_batch_thr:.1f}x higher")


In [ ]:
def to_3d_shap(sv, n_samples, n_features, n_classes):
    """Ενοποιεί την έξοδο SHAP σε 3D array σχήματος (N, F, C)."""
    if isinstance(sv, list):
        arr = np.stack([np.asarray(x, dtype=np.float64) for x in sv], axis=-1)
    else:
        arr = np.asarray(sv, dtype=np.float64)
    if arr.shape == (n_classes, n_samples, n_features):
        arr = np.transpose(arr, (1, 2, 0))
    assert arr.shape == (n_samples, n_features, n_classes), f"Unexpected SHAP shape: {arr.shape}"
    return arr

rng = np.random.default_rng(42)

# 1. Δημιουργία class-balanced δείγματος εξήγησης από το Training Set (έως 80 ροές/κλάση, 0 test leakage)
MAX_PER_CLASS = 80
exp_indices = []
for c in range(NUM_CLASSES):
    c_idx = np.where(y_train == c)[0]
    take = min(MAX_PER_CLASS, len(c_idx))
    chosen = rng.choice(c_idx, size=take, replace=False)
    exp_indices.append(chosen)
exp_indices = np.concatenate(exp_indices)
X_exp = np.ascontiguousarray(X_train[exp_indices])
y_exp = y_train[exp_indices]

# 2. Στρωματοποιημένο background dataset 100 ροών για το DeepExplainer (με τουλάχιστον 1 δείγμα ανά κλάση)
bg_indices = []
for c in range(NUM_CLASSES):
    c_idx = np.where(y_tr_mlp == c)[0]
    # 1 δείγμα εγγυημένο για κάθε σπάνια κλάση + αναλογική κατανομή των υπολοίπων
    bg_indices.append(rng.choice(c_idx, size=1, replace=False))
remaining_bg = 100 - NUM_CLASSES
all_pool = np.setdiff1d(np.arange(len(X_tr_mlp)), np.concatenate(bg_indices))
bg_indices.append(rng.choice(all_pool, size=remaining_bg, replace=False))
bg_indices = np.concatenate(bg_indices)
X_bg_t = torch.from_numpy(np.ascontiguousarray(X_tr_mlp[bg_indices])).to(device)

print(f"Balanced explanation set: {len(X_exp)} flows across all {NUM_CLASSES} classes | MLP background: {len(X_bg_t)} flows")

# --- RF Multiclass TreeSHAP ---
t0 = time.perf_counter()
rf_explainer = shap.TreeExplainer(rf_mc)
rf_sv_raw = rf_explainer.shap_values(X_exp, check_additivity=True)
rf_shap_time = time.perf_counter() - t0
rf_sv = to_3d_shap(rf_sv_raw, len(X_exp), len(features), NUM_CLASSES)

# Έλεγχος προσθετικότητας RF
rf_probs_exp = rf_mc.predict_proba(X_exp)
rf_base_vals = np.asarray(rf_explainer.expected_value, dtype=np.float64).ravel()
rf_add_err = float(np.max(np.abs((rf_base_vals[None, :] + rf_sv.sum(axis=1)) - rf_probs_exp)))
print(f"RF  TreeExplainer: {len(X_exp)} flows in {rf_shap_time:.2f}s | shape={rf_sv.shape} | max additivity err={rf_add_err:.2e}")

# --- MLP Multiclass DeepSHAP ---
mlp_mc.eval()
t0 = time.perf_counter()
mlp_explainer = shap.DeepExplainer(mlp_mc, X_bg_t)
mlp_sv_raw = mlp_explainer.shap_values(torch.from_numpy(X_exp).to(device), check_additivity=True)
mlp_shap_time = time.perf_counter() - t0
mlp_sv = to_3d_shap(mlp_sv_raw, len(X_exp), len(features), NUM_CLASSES)

with torch.inference_mode():
    mlp_logits_exp = mlp_mc(torch.from_numpy(X_exp).to(device)).cpu().numpy()
mlp_base_vals = np.asarray(mlp_explainer.expected_value, dtype=np.float64).ravel()
mlp_add_err = float(np.max(np.abs((mlp_base_vals[None, :] + mlp_sv.sum(axis=1)) - mlp_logits_exp)))
print(f"MLP DeepExplainer: {len(X_exp)} flows in {mlp_shap_time:.2f}s | shape={mlp_sv.shape} | max additivity err={mlp_add_err:.2e}")

# --- Υπολογισμός Per-Class Normalized SHAP Share (%) ---
# Για κάθε κλάση c, υπολογίζουμε mean(|SHAP[:, :, c]|) πάνω στα δείγματα της κλάσης c (ώστε να βλέπουμε τι χαρακτηρίζει κάθε επίθεση)
rf_per_class_pct = np.zeros((NUM_CLASSES, len(features)), dtype=np.float64)
mlp_per_class_pct = np.zeros((NUM_CLASSES, len(features)), dtype=np.float64)

for c in range(NUM_CLASSES):
    mask = (y_exp == c)
    rf_c = np.mean(np.abs(rf_sv[mask, :, c]), axis=0)
    mlp_c = np.mean(np.abs(mlp_sv[mask, :, c]), axis=0)
    rf_per_class_pct[c] = 100.0 * rf_c / (rf_c.sum() + 1e-12)
    mlp_per_class_pct[c] = 100.0 * mlp_c / (mlp_c.sum() + 1e-12)

rf_pc_df = pd.DataFrame(rf_per_class_pct, index=CLASS_NAMES, columns=features)
mlp_pc_df = pd.DataFrame(mlp_per_class_pct, index=CLASS_NAMES, columns=features)
rf_pc_df.to_csv("results/q6_shap_per_class_rf.csv")
mlp_pc_df.to_csv("results/q6_shap_per_class_mlp.csv")

# Global macro-averaged SHAP share across all 11 classes (ισότιμη στάθμιση όλων των τύπων επιθέσεων)
rf_global_pct = rf_per_class_pct.mean(axis=0)
mlp_global_pct = mlp_per_class_pct.mean(axis=0)

shap_cmp_df = pd.DataFrame({
    "Feature": features,
    "RF_SHAP_Share_Pct": rf_global_pct,
    "MLP_SHAP_Share_Pct": mlp_global_pct,
    "Mean_Share_Pct": 0.5 * (rf_global_pct + mlp_global_pct),
})
shap_cmp_df["RF_Rank"] = shap_cmp_df["RF_SHAP_Share_Pct"].rank(ascending=False, method="min").astype(int)
shap_cmp_df["MLP_Rank"] = shap_cmp_df["MLP_SHAP_Share_Pct"].rank(ascending=False, method="min").astype(int)
shap_cmp_df = shap_cmp_df.sort_values("Mean_Share_Pct", ascending=False).reset_index(drop=True)
shap_cmp_df.to_csv("results/q6_shap_global_comparison.csv", index=False)

rho_global, p_global = spearmanr(shap_cmp_df["RF_SHAP_Share_Pct"], shap_cmp_df["MLP_SHAP_Share_Pct"])
top10_rf = set(shap_cmp_df.nsmallest(10, "RF_Rank")["Feature"])
top10_mlp = set(shap_cmp_df.nsmallest(10, "MLP_Rank")["Feature"])
top15_rf = set(shap_cmp_df.nsmallest(15, "RF_Rank")["Feature"])
top15_mlp = set(shap_cmp_df.nsmallest(15, "MLP_Rank")["Feature"])
jacc10 = len(top10_rf & top10_mlp) / len(top10_rf | top10_mlp)
jacc15 = len(top15_rf & top15_mlp) / len(top15_rf | top15_mlp)

print(f"\n=== CROSS-MODEL EXPLAINABILITY AGREEMENT (RF TreeSHAP vs MLP DeepSHAP) ===")
print(f"Spearman rank correlation (all 44 features): rho = {rho_global:.4f} (p = {p_global:.2e})")
print(f"Top-10 Overlap: {len(top10_rf & top10_mlp)}/10 features (Jaccard = {jacc10:.3f}) -> {sorted(top10_rf & top10_mlp)}")
print(f"Top-15 Overlap: {len(top15_rf & top15_mlp)}/15 features (Jaccard = {jacc15:.3f})")

print("\n=== TOP 15 GLOBAL MULTICLASS FEATURES (MACRO-AVERAGED ACROSS 11 CLASSES) ===")
print(shap_cmp_df.head(15)[["Feature", "RF_SHAP_Share_Pct", "RF_Rank", "MLP_SHAP_Share_Pct", "MLP_Rank"]].round(3).to_string(index=False))

print("\n=== PER-ATTACK TOP-3 DRIVING FEATURES (RF TreeSHAP vs MLP DeepSHAP) ===")
for c, cname in enumerate(CLASS_NAMES):
    rf_t3 = [features[i] for i in np.argsort(rf_per_class_pct[c])[::-1][:3]]
    mlp_t3 = [features[i] for i in np.argsort(mlp_per_class_pct[c])[::-1][:3]]
    print(f"  {cname:18s} | RF: {', '.join(rf_t3):52s} | MLP: {', '.join(mlp_t3)}")

# --- XAI Inference Latency Benchmark (RF TreeExplainer vs MLP DeepExplainer) ---
XAI_BATCH_SIZE = 64

# RF single-flow & batch XAI
for _ in range(5):
    rf_explainer.shap_values(X_test[:1], check_additivity=False)
rf_xai_single = []
for i in range(100):
    ts = time.perf_counter_ns()
    rf_explainer.shap_values(X_test[i:i+1], check_additivity=False)
    rf_xai_single.append((time.perf_counter_ns() - ts) / 1_000_000)

rf_xai_batch_t = []
for _ in range(3):
    ts = time.perf_counter()
    rf_explainer.shap_values(X_test[:XAI_BATCH_SIZE], check_additivity=False)
    rf_xai_batch_t.append(time.perf_counter() - ts)
rf_xai_med_ms = float(np.median(rf_xai_single))
rf_xai_p95_ms = float(np.percentile(rf_xai_single, 95))
rf_xai_batch_thr = float(XAI_BATCH_SIZE / np.median(rf_xai_batch_t))

# MLP single-flow & batch XAI
for _ in range(3):
    mlp_explainer.shap_values(X_test_t[:1], check_additivity=False)
mlp_xai_single = []
for i in range(50):
    ts = time.perf_counter_ns()
    mlp_explainer.shap_values(X_test_t[i:i+1], check_additivity=False)
    mlp_xai_single.append((time.perf_counter_ns() - ts) / 1_000_000)

mlp_xai_batch_t = []
for _ in range(3):
    ts = time.perf_counter()
    mlp_explainer.shap_values(X_test_t[:XAI_BATCH_SIZE], check_additivity=False)
    mlp_xai_batch_t.append(time.perf_counter() - ts)
mlp_xai_med_ms = float(np.median(mlp_xai_single))
mlp_xai_p95_ms = float(np.percentile(mlp_xai_single, 95))
mlp_xai_batch_thr = float(XAI_BATCH_SIZE / np.median(mlp_xai_batch_t))

print("\n=== 11-CLASS XAI LATENCY BENCHMARK ===")
print(f"RF  TreeExplainer: Single Med={rf_xai_med_ms:.3f} ms (P95={rf_xai_p95_ms:.3f} ms) | Total Inf+XAI={rf_inf_med_ms+rf_xai_med_ms:.3f} ms ({(rf_inf_med_ms+rf_xai_med_ms)/rf_inf_med_ms:.2f}x) | Batch XAI Thr={rf_xai_batch_thr:.1f} flows/s")
print(f"MLP DeepExplainer: Single Med={mlp_xai_med_ms:.3f} ms (P95={mlp_xai_p95_ms:.3f} ms) | Total Inf+XAI={mlp_inf_med_ms+mlp_xai_med_ms:.3f} ms ({(mlp_inf_med_ms+mlp_xai_med_ms)/mlp_inf_med_ms:.1f}x) | Batch XAI Thr={mlp_xai_batch_thr:.1f} flows/s")

# Αποθήκευση συγκεντρωτικού πίνακα σύγκρισης (Classification + Operational + XAI)
rf_comb_thr = 1000.0 / (rf_amort_ms + 1000.0 / rf_xai_batch_thr)
mlp_comb_thr = 1000.0 / (mlp_amort_ms + 1000.0 / mlp_xai_batch_thr)

mc_comparison = pd.DataFrame([
    {
        **rf_metrics,
        "Train_Time_s": rf_train_time,
        "Model_Size_KB": rf_size_kb,
        "Inf_Median_ms": rf_inf_med_ms,
        "Inf_P95_ms": rf_inf_p95_ms,
        "Inf_Batch_Throughput_fps": rf_batch_thr,
        "XAI_Method": "TreeExplainer",
        "XAI_Median_ms": rf_xai_med_ms,
        "XAI_P95_ms": rf_xai_p95_ms,
        "Total_Inf_Plus_XAI_ms": rf_inf_med_ms + rf_xai_med_ms,
        "XAI_Slowdown_Factor": (rf_inf_med_ms + rf_xai_med_ms) / rf_inf_med_ms,
        "XAI_Batch_Throughput_fps": rf_xai_batch_thr,
        "Combined_Batch_Throughput_fps": rf_comb_thr,
    },
    {
        **mlp_metrics,
        "Train_Time_s": mlp_train_time,
        "Model_Size_KB": mlp_size_kb,
        "Inf_Median_ms": mlp_inf_med_ms,
        "Inf_P95_ms": mlp_inf_p95_ms,
        "Inf_Batch_Throughput_fps": mlp_batch_thr,
        "XAI_Method": "DeepExplainer (bg=100)",
        "XAI_Median_ms": mlp_xai_med_ms,
        "XAI_P95_ms": mlp_xai_p95_ms,
        "Total_Inf_Plus_XAI_ms": mlp_inf_med_ms + mlp_xai_med_ms,
        "XAI_Slowdown_Factor": (mlp_inf_med_ms + mlp_xai_med_ms) / mlp_inf_med_ms,
        "XAI_Batch_Throughput_fps": mlp_xai_batch_thr,
        "Combined_Batch_Throughput_fps": mlp_comb_thr,
    },
])
mc_comparison.to_csv("results/q6_multiclass_model_comparison.csv", index=False)

summary_q6 = {
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "mlp_architecture": "44 -> 128 (ReLU, Drop 0.2) -> 64 (ReLU, Drop 0.2) -> 32 (ReLU) -> 11",
    "mlp_class_weight_alpha": ALPHA,
    "shap_balanced_sample_size": int(len(X_exp)),
    "rf_max_additivity_error": rf_add_err,
    "mlp_max_additivity_error": mlp_add_err,
    "shap_spearman_rho": float(rho_global),
    "shap_spearman_pvalue": float(p_global),
    "top10_shared_features": sorted(top10_rf & top10_mlp),
    "top10_jaccard": float(jacc10),
    "top15_jaccard": float(jacc15),
    "models": mc_comparison.to_dict(orient="records"),
}
with open("results/q6_multiclass_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_q6, f, indent=2, ensure_ascii=False)

print("\nSaved: results/q6_multiclass_model_comparison.csv & results/q6_multiclass_summary.json")


In [ ]:
# --- Plot 4: Global Multiclass SHAP Comparison (Top 15 Bar + Rank Agreement Scatter) ---
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))

top15 = shap_cmp_df.head(15).iloc[::-1]
y_p = np.arange(len(top15))
bh = 0.38
axes[0].barh(y_p + bh/2, top15["RF_SHAP_Share_Pct"], height=bh, label="RF (TreeExplainer)", color="#1f77b4")
axes[0].barh(y_p - bh/2, top15["MLP_SHAP_Share_Pct"], height=bh, label="MLP (DeepExplainer)", color="#ff7f0e")
axes[0].set_yticks(y_p)
axes[0].set_yticklabels(top15["Feature"], fontsize=9.5)
axes[0].set_xlabel("Μέσο Ποσοστό SHAP Συνεισφοράς στις 11 Κλάσεις (%)")
axes[0].set_title("Top 15 Παγκόσμια Χαρακτηριστικά (RF vs MLP)", fontweight="bold")
axes[0].legend(loc="lower right", frameon=True)
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

axes[1].scatter(shap_cmp_df["RF_Rank"], shap_cmp_df["MLP_Rank"], s=55, alpha=0.85, color="#2ca02c", edgecolor="black", linewidth=0.5)
axes[1].plot([1, 44], [1, 44], linestyle="--", color="gray", alpha=0.7, label="Τέλεια συμφωνία (y = x)")
for _, r in shap_cmp_df.head(8).iterrows():
    axes[1].annotate(r["Feature"], (r["RF_Rank"], r["MLP_Rank"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
axes[1].set_xlabel("Κατάταξη RF TreeExplainer (1 = σημαντικότερο)")
axes[1].set_ylabel("Κατάταξη MLP DeepExplainer (1 = σημαντικότερο)")
axes[1].set_title(f"Συμφωνία Κατάταξης Features (Spearman ρ = {rho_global:.3f}, Top-10 Jaccard = {jacc10:.2f})", fontweight="bold")
axes[1].set_xlim(0, 46)
axes[1].set_ylim(0, 46)
axes[1].legend(loc="lower right", frameon=True)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("results/q6_shap_global_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 5: Side-by-Side 11x15 Per-Class Attack Signature Heatmaps ---
top15_feats = shap_cmp_df.head(15)["Feature"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(19, 6.8), sharey=True)

vmax_hm = float(max(rf_pc_df[top15_feats].max().max(), mlp_pc_df[top15_feats].max().max()))
vmax_hm = min(vmax_hm, 45.0)

sns.heatmap(
    rf_pc_df[top15_feats],
    annot=True, fmt=".1f", cmap="YlGnBu", vmin=0, vmax=vmax_hm,
    ax=axes[0], annot_kws={"size": 7.5}, linewidths=0.3, cbar_kws={"label": "SHAP Share (%)"}
)
axes[0].set_title("Random Forest (TreeExplainer) — Υπογραφές Features ανά Επίθεση (%)", fontsize=11.5, fontweight="bold")
axes[0].set_xlabel("Top 15 Χαρακτηριστικά")
axes[0].set_ylabel("Κλάση Κίνησης / Επίθεσης")
axes[0].tick_params(axis="x", rotation=40)

sns.heatmap(
    mlp_pc_df[top15_feats],
    annot=True, fmt=".1f", cmap="YlOrRd", vmin=0, vmax=vmax_hm,
    ax=axes[1], annot_kws={"size": 7.5}, linewidths=0.3, cbar_kws={"label": "SHAP Share (%)"}
)
axes[1].set_title("Multiclass MLP (DeepExplainer) — Υπογραφές Features ανά Επίθεση (%)", fontsize=11.5, fontweight="bold")
axes[1].set_xlabel("Top 15 Χαρακτηριστικά")
axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.savefig("results/q6_shap_per_class_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 6: Local Side-by-Side Explanations on 2 Unseen Test Attack Flows ---
scaler = joblib.load(scaler_path)["scaler"]

# Επιλογή 2 αθέατων επιθέσεων από το X_test που ταξινομήθηκαν σωστά ΚΑΙ από τα δύο μοντέλα
local_targets = [2, 6]  # 2 = SSH-BruteForce, 6 = DoS-Slowloris
fig, axes = plt.subplots(2, 2, figsize=(15.5, 9.2))

for row_i, target_cls in enumerate(local_targets):
    cname = CLASS_NAMES[target_cls]
    cand = np.where((y_test == target_cls) & (rf_preds == target_cls) & (mlp_preds == target_cls))[0]
    sample_idx = int(cand[0])
    x_s = X_test[sample_idx:sample_idx+1]
    x_raw = np.round(scaler.inverse_transform(x_s).ravel(), 2)

    rf_p = float(rf_mc.predict_proba(x_s)[0, target_cls])
    with torch.inference_mode():
        mlp_p = float(torch.softmax(mlp_mc(torch.from_numpy(x_s).to(device)), dim=1)[0, target_cls].cpu().item())

    rf_loc = to_3d_shap(rf_explainer.shap_values(x_s, check_additivity=True), 1, len(features), NUM_CLASSES)[0, :, target_cls]
    mlp_loc = to_3d_shap(mlp_explainer.shap_values(torch.from_numpy(x_s).to(device), check_additivity=True), 1, len(features), NUM_CLASSES)[0, :, target_cls]

    for col_i, (sv_vec, model_lbl, unit_lbl, prob_val) in enumerate([
        (rf_loc, "RF TreeExplainer", "ΔP(class)", rf_p),
        (mlp_loc, "MLP DeepExplainer", "ΔLogit(class)", mlp_p),
    ]):
        ax = axes[row_i, col_i]
        top_idx = np.argsort(np.abs(sv_vec))[::-1][:8][::-1]
        vals = sv_vec[top_idx]
        lbls = [f"{features[j]} = {x_raw[j]:g}" for j in top_idx]
        cols_bar = ["#d62728" if v >= 0 else "#1f77b4" for v in vals]
        ax.barh(np.arange(len(top_idx)), vals, color=cols_bar, alpha=0.88)
        ax.axvline(0, color="black", linewidth=0.9)
        ax.set_yticks(np.arange(len(top_idx)))
        ax.set_yticklabels(lbls, fontsize=8.8)
        ax.set_xlabel(f"SHAP Contribution ({unit_lbl})")
        ax.set_title(f"{cname} (test#{sample_idx}) — {model_lbl} | P={prob_val:.4f}", fontsize=10.5, fontweight="bold")
        ax.grid(axis="x", linestyle="--", alpha=0.45)

plt.suptitle("Τοπικές Εξηγήσεις SHAP στις Ίδιες Αθέατες Επιθέσεις: RF TreeExplainer vs MLP DeepExplainer", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("results/q6_shap_local_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 7: Operational & XAI Latency / Throughput Trade-off (RF vs MLP) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))

models_lbl = ["Random Forest\n(Multiclass)", "MLP\n(Multiclass)"]
xp = np.arange(2)
bw = 0.34

inf_vals = [rf_inf_med_ms, mlp_inf_med_ms]
tot_vals = [rf_inf_med_ms + rf_xai_med_ms, mlp_inf_med_ms + mlp_xai_med_ms]

b_inf = axes[0].bar(xp - bw/2, inf_vals, width=bw, label="Inference μόνο (Median ms)", color="#1f77b4")
b_tot = axes[0].bar(xp + bw/2, tot_vals, width=bw, label="Inference + 11-Class SHAP (ms)", color="#d62728")
axes[0].set_yscale("log")
axes[0].set_xticks(xp)
axes[0].set_xticklabels(models_lbl, fontsize=10)
axes[0].set_ylabel("Single-Flow Latency (ms — log scale)")
axes[0].set_title("Single-Flow Latency: Inference vs Inference + 11-Class XAI", fontweight="bold")
for b in list(b_inf) + list(b_tot):
    v = b.get_height()
    axes[0].text(b.get_x() + b.get_width()/2, v * 1.22, f"{v:.3f} ms", ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[0].set_ylim(0.01, max(tot_vals) * 4.5)
axes[0].legend(loc="upper left", frameon=True)
axes[0].grid(axis="y", which="both", linestyle="--", alpha=0.45)

thr_inf = [rf_batch_thr, mlp_batch_thr]
thr_xai = [rf_comb_thr, mlp_comb_thr]
b_ti = axes[1].bar(xp - bw/2, thr_inf, width=bw, label="Inference Batch Throughput (flows/s)", color="#2ca02c")
b_tx = axes[1].bar(xp + bw/2, thr_xai, width=bw, label="Inference + Inline 11-Class XAI (flows/s)", color="#ff7f0e")
axes[1].set_yscale("log")
axes[1].set_xticks(xp)
axes[1].set_xticklabels(models_lbl, fontsize=10)
axes[1].set_ylabel("Throughput (flows/sec — log scale)")
axes[1].set_title("Batch Throughput: Χωρίς vs Με Inline 11-Class XAI", fontweight="bold")
for b in list(b_ti) + list(b_tx):
    v = b.get_height()
    lbl = f"{v:,.0f}" if v >= 100 else f"{v:.1f}"
    axes[1].text(b.get_x() + b.get_width()/2, v * 1.25, lbl, ha="center", va="bottom", fontsize=9, fontweight="bold")
axes[1].set_ylim(1, max(thr_inf) * 6.0)
axes[1].legend(loc="upper left", frameon=True)
axes[1].grid(axis="y", which="both", linestyle="--", alpha=0.45)

plt.tight_layout()
plt.savefig("results/q6_operational_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
